### Support Vector Machines (SVM) & Its Sklearn Implementations

#### 1. Basic Intuition: SVM Kya Hai?
Maan lijiye aap ek map par hain jahan do alag-alag squads (teams) ke log khade hain. Aapko un dono squads ke beech ek aisi border (diwaar) khadi karni hai, jo dono squads se **sabse zyada door** ho. Taaki agar kal ko koi naya player drop ho, toh aap border dekh kar turant bata sakein ki wo kis squad ke area me land hua hai.

Is "diwaar" ya "border" ko hi Machine Learning me **Hyper-plane** kehte hain, aur aisi best boundary dhoondhne ke technique ko **Support Vector Machine (SVM)** kehte hain.

* **Support Vectors:** Yeh wo sabse aage khade hue players (data points) hain jo border ke ekdam kareeb hote hain. Inhi ke position ke basis par poori border line fix hoti hai. Agar baaki peeche wale players hat bhi jayein, toh border par koi farq nahi padega.
* **Margin:** Border (Hyperplane) aur sabse kareebi players (Support vectors) ke beech ka jo khali space (gap) hota hai, use Margin kehte hain. SVM hamesha is gap ko bada karne (maximize) ki koshish karta hai.

---

#### 2. Sklearn me 3 Methods (SVC, NuSVC, LinearSVC)
Aapke notes me likha hai ki Sklearn (Python library) in 3 tariko se SVM chalati hai. In teeno me kya fark hai, aaiye ekdam aasan bhasha me samjhein:

**A. SVC (Support Vector Classification)**
* **Kya hai:** Yeh default aur sabse popular method hai. Yeh straight line (Linear) ke sath-sath curved lines (Non-linear/RBF kernels) bhi bana sakta hai agar data complex ho.
* **Backend Engine:** Yeh `libsvm` library par based hai.
* **Use case:** Jab data medium size ka ho (e.g., 10,000 rows) aur boundary complex ho.

**B. NuSVC (Nu-Support Vector Classification)**
* **Kya hai:** Yeh bilkul SVC ka judwa bhai hai. Fark sirf itna hai ki SVC me margin set karne ke liye `C` naam ka parameter use hota hai, aur NuSVC me `nu` (ek value 0 se 1 ke beech) use hoti hai. `nu` direct control karta hai ki aap model me kitni errors (galatiyan) bardasht kar sakte hain.
* **Backend Engine:** Yeh bhi `libsvm` par chalata hai.

**C. LinearSVC**
* **Kya hai:** Yeh sirf aur sirf "Seedhi Line" (Linear boundary) banata hai. Isme complex (curved) boundaries banane ki shamta (Kernels) nahi hoti. 
* **Backend Engine:** Yeh `liblinear` par based hai, isliye yeh SVC se bohot zyada FAST hai.
* **Modern Context:** Text data (NLP) ya bohot massive datasets (Millions of rows) me dimensions itni zyada hoti hain ki straight line hi sabse best kaam karti hai. Aisi jagah aajkal data scientists `SVC(kernel='linear')` ki jagah directly `LinearSVC` use karte hain memory aur time bachane ke liye.

---

#### 3. The Mathematics (Advanced: Equation of Hyperplane)
Aaiye math me is border (Hyperplane) ko samajhte hain.

Ek straight line ki basic equation hoti hai $y = mx + c$. Vector math (advance dimensions) me hum isko is tarah likhte hain:
$$w \cdot x + b = 0$$

Jahan:
* $w$ = Weight vector (Jo batata hai line ka angle kaisa hoga).
* $x$ = Aapka input data point.
* $b$ = Bias (Jo line ko aage-peeche shift karta hai).

**Margins ki equation:**
Jo points (support vectors) boundary ke ekdam pass hote hain, unke liye equation hoti hai:
Class 1 ke liye: $w \cdot x + b = 1$
Class 0 ke liye: $w \cdot x + b = -1$

**Model ka Asli Goal (Optimization Problem):**
In dono margins ke beech ka distance mathematically $\frac{2}{||w||}$ hota hai. 
Is distance ko sabse bada (maximize) karne ke liye, SVM algorithm ko $||w||$ ko sabse chhota (minimize) karna hota hai. 

Isliye mathematical objective function yeh banta hai:
$$Minimize: \frac{1}{2} ||w||^2$$
$$Subject\_to: y_i(w \cdot x_i + b) \ge 1$$
(Yani har point apne sahi area me ho aur boundary se door rahe).

In [12]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X,y = load_iris(return_X_y=True)
x_train , x_test , y_train , y_test = train_test_split(X,y , test_size=0.3 , random_state=8)

### SVC (Support Vector CLassifier)

In [11]:
from sklearn.svm import SVC
svc_classifier = SVC()
svc_classifier.fit(x_train , y_train)
y_pred = svc_classifier.predict(x_test)
print( mean_squared_error(y_test , y_pred) )

0.044444444444444446


### Support Vector Classifiers (SVC): Advanced Tuning of 'C' and 'Kernels'

#### 1. Basic Intuition (The Ground Reality)
Machine learning me classification ka matlab hai data ke beech ek boundary banana. Maan lijiye hum ek system bana rahe hain jo players ke in-game movement aur kill stats (jaise F/D ratio aur headshot accuracy) dekh kar unhe 'Competitive/Esports Player' aur 'Casual Player' me classify karta hai. 

Is classification me boundary banate waqt do sabse badi problems aati hain:
1. **Errors ko kitna bardasht karein?** (Kya ek casual player jo by-chance ek match me pro wale stats le aaya, use model pro maan le? Ya system strict rahe?) Ise control karta hai **Regularization Parameter (`C`)**.
2. **Boundary ka shape kaisa ho?** (Kya in players ko ek seedhi line se alag kiya ja sakta hai, ya koi complex curve chahiye?) Ise decide karta hai **`Kernel`**.

---

#### 2. Core Concepts & Architecture (Step-by-Step Breakdown)

**A. Regularization Parameter (`C`)**
`C` ek hyperparameter hai jo model ko batata hai ki misclassification (galat prediction) par kitni penalty (saza) lagani hai. Default value iski `1.0` hoti hai (jo ek balanced approach hai).

* **Inversely Proportional Strength:** Yeh sabse critical point hai. `C` ki value jitni **badi** hogi, regularization ki taqat utni **kam** hogi. 
    * **High C (e.g., C=100.0):** Model bohot strict ho jata hai. Wo training data me ek bhi point ko galat classify nahi karna chahta. Boundary bohot complex aur zig-zag banegi, jisse **Overfitting** (data ko ratne) ka khatra hota hai.
    * **Low C (e.g., C=0.01):** Model thoda relaxed hota hai. Regularization high hota hai. Boundary smooth banti hai (generalization achha hota hai), par thodi bohot galtiyan model training me hi accept kar leta hai (Soft Margin).
* **Strictly Positive:** `C` ek float value hoti hai jo hamesha $0$ se badi honi chahiye. Negative saza ka math me koi existance nahi hota.
* **Squared L2 Penalty:** SVC under-the-hood weights ko handle karne ke liye L2 norm ($||w||^2$) penalty lagata hai. Iska matlab hai ki parameters ka square karke penalty di jati hai, jisse koi ek single feature model par haavi nahi ho pata.

**B. The `Kernel` Trick**
Jab aapke features complex hote hain aur unhe seedhi line (linear plane) se alag nahi kiya ja sakta, toh hum data ko mathematically higher dimensions (jaise 2D se 3D ya infinite-D) me project karte hain. Wahan ek seedhi plane draw karke unhe alag kiya jata hai. Is technique ko Kernel Trick kehte hain.

* **`'rbf'` (Radial Basis Function):** Yeh SVC ka default aur sabse powerful kernel hai. Yeh data ko infinite dimensions me le jata hai aur circular/island jaisi boundaries banata hai.
* **`'linear'`:** Yeh sirf ek seedhi line ya flat plane banata hai. Agar data me bohot saare columns (high dimensionality) pehle se hain, toh linear best aur fastest kaam karta hai.
* **`'poly'` (Polynomial):** Yeh parabolic ya cubic curves banata hai. Agar aap isko select karte hain, toh aapko algorithm ko `degree` batani padti hai (e.g., `degree=3` for cubic curve).
* **`'sigmoid'`:** Yeh neural networks ke activation function jaisa curve banata hai, par industry me aajkal kam use hota hai.
* **`'precomputed'`:** Agar aapke paas raw data nahi hai, balki aapne khud data points ke beech ke distances ka matrix calculate kar rakha hai, toh use direct isme pass kar sakte hain.

---

#### 3. Advanced Mathematics (Behind the Scenes)

**Optimization of 'C' (Soft Margin SVM):**
SVC model ka main engine is mathematical objective function ko minimize karta hai:

$$Minimize: \frac{1}{2} ||w||^2 + C \sum_{i=1}^{m} \xi_i$$

Is equation me kya ho raha hai?
* $\frac{1}{2} ||w||^2$: Yeh hamara L2 Penalty term hai jo margin ko bada rakhne ki koshish karta hai.
* $\xi_i$ (Xi / Slack Variables): Yeh measure karte hain ki koi point boundary cross karke apne galat side par kitna andar tak ghusa hua hai (Error distance).
* $C$: Yeh trade-off banata hai. Agar $C$ bohot bada hai, toh equation $\sum \xi_i$ (errors) ko zero karne pe puri taqat laga degi (strict boundary). Agar $C$ chhota hai, toh equation margin $\frac{1}{2} ||w||^2$ ko focus karegi.

**The RBF Kernel Equation:**
RBF actual me data ko naye dimension me nahi bhejta, wo direct do points ke beech ki similarity nikal leta hai:
$$K(x, x') = \exp(-\gamma ||x - x'||^2)$$
Jahan $\gamma$ (gamma) distance ka spread control karta hai.

---

#### 4. Real-World Industry Use-Case
Industry me SVM ka bohot heavy use **Fraud Detection and Anomaly Classification** me hota hai. 
Example ke liye, Income Tax e-filing portals par daily lakho returns aate hain. Jo log proper channels se e-filing karte hain unka behavior normal cluster me hota hai, par jo bots ya bad actors automated fake returns file karte hain unke patterns (time taken, IP anomalies) alag hote hain. 
Aise highly complex financial data me hum straight line se fraud nahi pakad sakte. Wahan hum `SVC(kernel='rbf')` use karte hain. Ek proper pipeline banayi jati hai jahan `C` parameter ko bohot carefully tune kiya jata hai taaki fraud pakadne me strictness bhi rahe, par kisi genuine user ka registration block na ho jaye.

---

#### 5. Modern Implementation (Production Grade Code)
Corporate production code me hum kabhi direct raw data `SVC()` ko nahi dete. Hum hamesha `StandardScaler` use karte hain kyunki SVM distance-based algorithm hai aur unscaled data isko completely fail kar dega. Hyperparameters dhoondhne ke liye `GridSearchCV` standard practice hai.


#### 4. Training (Assuming X_train and y_train are pre-defined)
#### grid_search.fit(X_train, y_train)

#### 5. Extracting the optimal parameters
#### print(f"Best Hyperparameters found: {grid_search.best_params_}")

In [13]:
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline

# 1. Pipeline Creation: Scaling is strictly MANDATORY for SVC
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(class_weight='balanced')) # Imbalanced data ke liye balanced
])

# 2. Hyperparameter Grid: Testing different combinations strictly
param_grid = [
    # Linear kernel ke liye alag C values test karna
    {'classifier__kernel': ['linear'], 'classifier__C': [0.1, 1.0, 10.0]},
    
    # RBF kernel (Default) ke liye C test karna
    {'classifier__kernel': ['rbf'], 'classifier__C': [0.1, 1.0, 10.0, 100.0]},
    
    # Poly kernel ke liye C aur Degree dono explicitly set karna
    {'classifier__kernel': ['poly'], 'classifier__C': [1.0], 'classifier__degree': [2, 3]}
]

# 3. Grid Search setup for best model extraction
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=5, # 5-Fold Cross Validation
    scoring='accuracy',
    n_jobs=-1 # Utilize all available CPU cores
)

### Advanced Kernel Parameters: `gamma` and `coef0`

#### 1. Basic Intuition (The Ground Reality)
Maan lijiye har data point ek Wi-Fi router hai jo apna signal chhod raha hai. 
* **`gamma` (Signal Range):** Yeh batata hai ki ek single data point ka asar (influence) kitni door tak jayega. 
  * **Low Gamma (Chhota Number):** Router ka signal bohot door tak ja raha hai. Boundaries bohot smooth aur broad banengi. (Sab connected lagenge).
  * **High Gamma (Bada Number):** Router ka signal sirf uske ekdam paas wale area me hai. Har point ka apna ek chhota sa "island" ban jayega. Isse boundary bohot sharp aur zig-zag banti hai.

* **`coef0` (Baseline Boost):** Yeh sirf Polynomial aur Sigmoid curves me use hota hai. Yeh basically ek extra constant value (bias) hai jo curve ko thoda shift karti hai taaki math calculations me chhote aur bade numbers ke beech balance bana rahe.

---

#### 2. Core Concepts & Architecture (Step-by-Step Breakdown)

**A. The `gamma` Parameter**
Yeh parameter 'rbf', 'poly', aur 'sigmoid' teeno kernels ke sath kaam karta hai. Isko set karne ke 3 tarike hote hain:

1. **`gamma = 'scale'` (The Modern Default & Best Practice):** Yeh sabse smart tarika hai. Yeh aapke data ke features aur unke spread (variance) ko dekh kar khud decide karta hai ki gamma kya hona chahiye. Isse over-fitting se bachne me bohot madad milti hai.
   * *Formula:* $\frac{1}{n\_features \times X.var()}$

2. **`gamma = 'auto'` (The Old Way):**
   Yeh purane Scikit-Learn versions ka default tha. Yeh sirf total columns (features) count karta hai aur data ke variance (spread) ko ignore kar deta hai. Isliye isko ab industry me kam use kiya jata hai.
   * *Formula:* $\frac{1}{n\_features}$

3. **Float Value (Manual Control):** Aap khud ek fixed number de sakte hain (jaise 0.001, 0.1, 10.0). Ise hum grid search se dhoondhte hain.

**B. The `coef0` Parameter**
Yeh ek independent term hai jo sirf `'poly'` aur `'sigmoid'` kernels me lagti hai. Default value $0.0$ hoti hai. Yeh curve ko space me move karne me madad karti hai, jisse model complex patterns ko zyada easily pakad pata hai.

---

#### 3. Advanced Mathematics (Behind the Scenes)

**The RBF Kernel Equation (`gamma` in action):**
$$K(x, x') = \exp(-\gamma ||x - x'||^2)$$
* Yahan $||x - x'||^2$ do points ke beech ka squared Euclidean distance hai.
* Agar $\gamma$ bohot bada hoga, toh exponential power bohot badi negative value ho jayegi, jisse result turant $0$ ho jayega. Yani sirf bohot paas wale points hi ek dusre ko influence kar payenge (creating tight islands).

**The Polynomial Kernel Equation (`gamma` and `coef0` in action):**
$$K(x, x') = (\gamma \langle x, x' \rangle + r)^d$$
* Yahan $r$ aapka **`coef0`** hai. 
* $d$ aapka `degree` hai.
* Agar $r = 0$ hai aur aapka data scale nahi kiya gaya hai, toh higher-degree polynomial direct explosion (infinite values) ki taraf ja sakta hai. $r$ ko adjust karne se lower-order (linear) aur higher-order (curved) terms ke beech balance banta hai.

---

#### 4. Real-World Industry Use-Case
**Medical Imaging (Tumor Classification):**
Maan lijiye MRI scans ke pixels se data features extract kiye gaye hain. Tumor ke pixels (features) bohot complex aur overlapped hote hain healthy tissues ke sath. Agar hum manual `gamma` dein toh model fail ho sakta hai kyunki har machine ka MRI variance alag hota hai. Aisi jagah hum hamesha `SVC(kernel='rbf', gamma='scale')` use karte hain. `'scale'` automatically data ki variance (lightness/darkness of image features) calculate karke perfect Wi-Fi range set kar deta hai, jisse model universally har hospital ke data par perfectly chalta hai.

---

#### 5. Modern Implementation (Production Grade Code)
Production me hum `gamma='scale'` ko default rakhte hain, par best results ke liye hum isko manual float values ke sath `GridSearchCV` me daal kar test karte hain.

---

In [14]:
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline

# 1. Scaling is absolute strictly required for RBF and Poly
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(class_weight='balanced'))
])

# 2. Hyperparameter tuning specifically for Gamma and Coef0
param_grid = [
    # RBF Kernel testing Gamma limits
    {
        'svm__kernel': ['rbf'],
        'svm__C': [1, 10, 100],
        'svm__gamma': ['scale', 'auto', 0.1, 0.01, 0.001] # Testing both strings and floats
    },
    # Polynomial Kernel testing Gamma and Coef0
    {
        'svm__kernel': ['poly'],
        'svm__degree': [2, 3],
        'svm__gamma': ['scale', 0.1, 1.0],
        'svm__coef0': [0.0, 1.0, 10.0] # Testing the bias term
    }
]

# 3. Running Grid Search to find perfect balance
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=5, 
    n_jobs=-1
)

# grid_search.fit(X_train, y_train)
# print("Best Kernel Parameters:", grid_search.best_params_)

### Demystifying NuSVC: The Mathematical Elegance of the $\nu$ (Nu) Parameter

#### 1. Basic Intuition (The Ground Reality)
Ab tak humne dekha ki SVC me model ko strict ya relax karne ke liye hum `C` parameter use karte the. Par as an engineer, `C` ke sath ek bohot badi practical problem hai: **`C` ki range infinite hai (0.001 se leke 10,000 ya usse bhi zyada).** Aapko kaise pata chalega ki aapke data ke liye $C=10$ best hai ya $C=100$? Yeh pure guesswork ban jata hai. 

Is problem ko solve karne ke liye **NuSVC** ($\nu$-SVC) banaya gaya. NuSVC ne `C` parameter ko hata kar ek naya parameter introduce kiya jise **$\nu$ (Nu)** kehte hain. 
Iska sabse bada fayda yeh hai ki yeh ek **Percentage / Fraction** hai jo hamesha $0$ se $1$ ke beech hota hai. 

Agar aap $\nu = 0.1$ set karte hain, toh aap model ko seedhe shabdon me ek mathematical instruction de rahe hain: 
* "Bhai, mere training data me maximum **10%** points ki galti (margin errors) main bardasht kar lunga."
* "Aur kam se kam **10%** points ko boundary set karne ke liye (support vectors) use karna."

---

#### 2. Core Concepts & Architecture (Step-by-Step Breakdown)

**The Dual Bounding Power of $\nu$** , 
$\nu$ ek akele parameter me do alag-alag limits (bounds) ko control karta hai:

1. **Upper Bound on Margin Errors (Maximum Galti):** Margin Error ka matlab sirf misclassified points nahi hota. Agar koi point sahi class me hai, par margin ke andar ghus gaya hai, toh wo bhi error ($\xi > 0$) kahlata hai. $\nu$ ensure karta hai ki total training data me se **$\nu$ fraction se zyada margin errors na hon**. 
   *(e.g., Data 1000 rows ka hai, $\nu = 0.05$ hai. Toh kisi bhi haalat me 50 se zyada points margin ke andar ya galat side nahi honge).*

2. **Lower Bound on Support Vectors (Minimum Anchors):**
   Support vectors wo points hote hain jo boundary define karte hain. $\nu$ ensure karta hai ki model **kam se kam $\nu$ fraction** data points ko support vectors banaye. 
   *(e.g., $\nu = 0.05$ matlab kam se kam 50 points boundary ko hold karke rakhenge. Yeh model ko stable banata hai taaki 1-2 outliers boundary ko kharab na kar dein).*

**Range and Default:**
* **Limits:** $\nu \in (0, 1]$. Yeh 0 nahi ho sakta kyunki math me division by zero ya boundary collapse ho jayega.
* **Default ($\nu = 0.5$):** Agar aap kuch set nahi karte, toh model koshish karega ki maximum 50% errors hon aur minimum 50% data points support vectors banein.

---

#### 3. Advanced Mathematics (Behind the Scenes)

NuSVC ka math standard SVC se thoda alag aur zyada smart hai. Standard SVC me margin ka size $2 / ||w||$ hota hai aur margin lines $1$ aur $-1$ par fix hoti hain.
NuSVC me hum margin ko ek variable bana dete hain, jise **$\rho$ (Rho)** kehte hain. Margin lines ab $\rho$ aur $-\rho$ par hoti hain, aur total margin $2\rho / ||w||$ ban jata hai.

**The Nu-SVC Optimization Function:**
$$Minimize: \frac{1}{2} ||w||^2 - \nu \rho + \frac{1}{m} \sum_{i=1}^{m} \xi_i$$

$$Subject\_to: y_i(w \cdot x_i + b) \ge \rho - \xi_i \quad \text{and} \quad \xi_i \ge 0, \rho \ge 0$$

Is equation me kya ho raha hai?
* $-\nu \rho$: Algorithm $\rho$ (margin) ko bada (maximize) karne ki koshish karta hai kyunki uske aage negative sign hai.
* $\frac{1}{m} \sum \xi_i$: Yeh total errors (slack) ko chhota karne ki koshish karta hai.
* $\nu$ parameter in dono takaton ke beech ek trade-off set karta hai. Agar $\nu$ bada hoga, toh $\rho$ ko bada karne par zyada focus hoga, jisse margin fail jayega aur errors zyada allow ho jayenge. 

**Proved Mathematical Theorem (Schölkopf et al., 2000):**
$$\text{Fraction of Margin Errors} \le \nu \le \text{Fraction of Support Vectors}$$

---

#### 4. Real-World Industry Use-Case
**Predictive Maintenance in Jet Engines (Sensor Anomaly):**
Aviation industry me jab jet engine ke sensors se heat aur vibration ka data aata hai, toh hum chahte hain ki engine failure ki boundary ekdam perfect bane. 
Wahan data scientists `SVC(C=10)` guess nahi karte. Wo jante hain ki historical data me exact 2% sensors ke noise/errors aate hi aate hain. Toh wo directly **`NuSVC(nu=0.02)`** use karte hain. Isse model automatically apna margin aise adjust karta hai ki sirf top 2% extreme anomalies hi boundary ko cross karein, aur model overfit hone se bach jaye.

---

#### 5. Modern Implementation (Production Grade Code)
Production me NuSVC ka use karte waqt ek sabse critical cheez hoti hai: **Infeasible $\nu$ Error**. Agar aap $\nu$ ko bohot bada (jaise 0.9) set kar denge, par aapka data aapas me bohot tightly mixed hai, toh mathematical optimization fail ho jayega (kisi constraint ka solution nahi milega) aur `ValueError` aayega. Isliye production code me isko `try-except` block me likhna best practice hai.

---

In [17]:
from sklearn.svm import NuSVC
NuSVC_classifier = NuSVC()

NuSVC_classifier.fit(x_train , y_train)

,"nu nu: float, default=0.5An upper bound on the fraction of margin errors (see :ref:`User Guide`) and a lower bound of the fraction of support vectors.Should be in the interval (0, 1].",0.5
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: {dict, 'balanced'}, default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one. The ""balanced"" mode uses the values of y to automaticallyadjust weights inversely proportional to class frequencies as``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [18]:
import numpy as np
from sklearn.svm import NuSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# 1. Pipeline creation with Mandatory Scaling
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', NuSVC(kernel='rbf', gamma='scale')) # Baaki parameters SVC jaise hote hain
])

# 2. Nu values grid - testing different percentage boundaries (e.g., 5%, 10%, 20%)
param_grid = {
    'classifier__nu': [0.05, 0.1, 0.2, 0.5]
}

# 3. Model setup
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=5, 
    scoring='f1_macro' # Imbalanced data ke liye best
)

# 4. Training with Safety Net for Infeasible Nu limits
# Man lijiye X_train, y_train available hain
"""
try:
    grid_search.fit(X_train, y_train)
    print(f"Best Nu parameter: {grid_search.best_params_['classifier__nu']}")
except ValueError as e:
    # Yeh tab trigger hoga agar aapne Nu itna bada de diya jo mathematically possible na ho
    print(f"Mathematical Optimization Failed: {e}")
    print("Solution: Try reducing the upper range of Nu in param_grid.")
"""

'\ntry:\n    grid_search.fit(X_train, y_train)\n    print(f"Best Nu parameter: {grid_search.best_params_[\'classifier__nu\']}")\nexcept ValueError as e:\n    # Yeh tab trigger hoga agar aapne Nu itna bada de diya jo mathematically possible na ho\n    print(f"Mathematical Optimization Failed: {e}")\n    print("Solution: Try reducing the upper range of Nu in param_grid.")\n'

### LinearSVC

### LinearSVC Under the Hood: Penalties, Losses, and the Dual Problem

#### 1. Basic Intuition (The Ground Reality)
Ab tak humne jo `SVC` dekha tha, wo `libsvm` engine par chalta hai, jo curves aur complex boundaries (kernels) banane me expert hai. Par jab humare paas text data hota hai jisme **lakho rows aur lakho columns (features)** hote hain, toh `SVC` bohot slow ho jata hai (RAM crash ho sakti hai).

Yahan entry hoti hai **`LinearSVC`** ki. Yeh `liblinear` engine par chalta hai. Ise ek aisi sports car samajh lijiye jisme sirf "Seedhe" (Linear) chalne ki shamta hai, par iski speed standard SVC se 100x zyada hoti hai. Is high-speed engine ko tune karne ke liye humare paas 4 main technical knobs hote hain: `penalty`, `loss`, `C`, aur `dual`.

---

#### 2. Core Concepts & Architecture (Step-by-Step Breakdown)

**A. The `penalty` Parameter (Saza ka Tareeqa)**
Penalty batata hai ki jab model overfit karne lage (features ke weights/coefficients bohot bade hone lagein), toh usko kaise control kiya jaye.
* **`l2` (Ridge / Default):** Yeh ek "Communist" approach hai. Yeh saare features ke weights ko thoda-thoda chhota (shrink) kar deta hai, par kisi ko zero nahi karta. Model saare features par thoda-thoda dhyan deta hai.
* **`l1` (Lasso / Sparse Vectors):** Yeh ek "Ruthless Boss" ki tarah hai. Jo features usko lagta hai ki less important hain, yeh unke weights ko **exact 0.0** kar deta hai. Isko bolte hain **Sparsity** lana. Agar 1000 features hain, toh L1 unme se 900 ko delete (0) kar dega aur sirf top 100 features se model chalayega.

**B. The `loss` Parameter (Galti ka Measure)**
Loss function calculate karta hai ki aapka point boundary se kitni door galat side par khada hai.
* **`hinge` (Standard SVM Loss):** Agar point 1 step galat hai, toh penalty 1. Agar 10 step galat hai, toh penalty 10. Yeh seedha (linear) kaam karta hai.
* **`squared_hinge` (Default for LinearSVC):** Yeh square karta hai. Agar point 1 step galat hai, penalty 1. Par agar 10 step galat hai, toh penalty $10^2 = 100$. Yeh badi galtiyon ko bohot bhari saza deta hai jisse optimization algorithm (gradient descent) zyada smoothly kaam karta hai.

**C. The Unsupported Combination (`l1` + `hinge`)**
Aapke notes me likha hai ki yeh dono ek sath kaam nahi karte. Kyun? Kyunki `l1` math me sharp corners banata hai (non-differentiable at 0) aur `hinge` bhi sharp V-shape banata hai. Jab algorithm calculus lagata hai in dono sharp corners ko ek sath solve karne ke liye, toh gradient (slope) calculate nahi ho pata aur algorithm crash ho jata hai. Isliye Scikit-Learn isko mana kar deta hai.

**D. The `dual` Parameter (The Mathematical Viewpoint)**
Yeh sabse critical optimization choice hai. Machine learning me har optimization problem ko do nazariye se solve kiya ja sakta hai:
* **Primal Problem:** Aap directly **Features (Columns)** ki equations solve kar rahe hain.
* **Dual Problem:** Aap features ko chhod kar directly **Data Points (Rows)** ke aapas ke relations solve kar rahe hain.

**E. `fit_intercept`**
Yeh batata hai ki kya equation $w \cdot x + b = 0$ me $b$ (bias/intercept) calculate karna hai ya nahi. Isko hamesha `True` chhodna chahiye, warna aapki boundary hamesha origin $(0,0)$ se pass hone ke liye majboor ho jayegi.

---

#### 3. Advanced Mathematics (Behind the Scenes)

**Loss Functions ka Math:**
* **Hinge Loss:** $L(y, \hat{y}) = \max(0, 1 - y_i(w \cdot x_i + b))$
* **Squared Hinge:** $L(y, \hat{y}) = (\max(0, 1 - y_i(w \cdot x_i + b)))^2$
(Aap dekh sakte hain ki squared hinge bas hinge loss ka mathematical square hai).

**The Rule of `dual=False` (Time Complexity Math):**
Notes me ek bohot tagdi line hai: *"When n_samples > n_features, prefer dual=False"*.
Iska mathematically proof time-complexity me chhipa hai:
* **Dual Problem ko solve karne ka time:** $\approx O(n\_features \times n\_samples^2)$
* **Primal Problem ko solve karne ka time:** $\approx O(n\_samples \times n\_features^2)$

Man lijiye aapke paas 10 Lakh Data Points (Rows/samples) hain aur sirf 10 Columns (features) hain. 
* Agar Dual chalaya: $10 \times (1,000,000)^2$ = Bohot bada number (Laptop hang).
* Agar Primal (`dual=False`) chalaya: $1,000,000 \times 10^2 = 100,000,000$ (Microseconds me solve!).
Isliye, jab bhi Data Rows $>$ Columns ho, aankh band karke `dual=False` set karein.

---

#### 4. Real-World Industry Use-Case
**Spam Email Detection System (NLP):**
Jab Gmail spam emails filter karta hai, toh text processing (TF-IDF) ke baad words ki dictionary itni badi ho jati hai ki columns (features) > 50,000 ho jate hain, aur emails (rows) millions me hote hain.
Wahan log `SVC` use nahi karte, wo `LinearSVC` use karte hain `penalty='l1'` ke sath. 
Kyun? Kyunki 50,000 words me se spam detect karne ke liye sirf "Lottery", "Free", "Click" jaise 500 words hi kaam ke hote hain. `l1` penalty baaki 49,500 words ke weights ko $0$ kar deti hai (Sparse vector). Isse production me memory bohot kam lagti hai aur prediction milliseconds me ho jati hai.

---

#### 5. Modern Implementation (Production Grade Code)
Corporate environment me jab bhi `LinearSVC` use hota hai, toh data engineer strict parameters aur scaling ka dhyan rakhta hai taaki convergence warnings na aayein.

---

In [21]:
from sklearn.svm import LinearSVC

LinearSVC_classifier = LinearSVC()
LinearSVC_classifier.fit(x_train , y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [22]:
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Dummy Large Dataset: 100,000 rows, 20 features (samples >> features)
X, y = make_classification(n_samples=100000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Production Pipeline Setup
# Rule followed: rows (100,000) > features (20), hence dual=False is strictly used.
pipeline = Pipeline([
    ('scaler', StandardScaler()), 
    ('clf', LinearSVC(
        penalty='l1',             # We want sparse weights (feature selection)
        loss='squared_hinge',     # Must be squared_hinge with l1 penalty
        dual=False,               # MANDATORY here because samples > features
        C=1.0,                    # Standard regularization
        tol=1e-4,                 # Tolerance for stopping criteria
        max_iter=10000            # Increased max_iter to prevent convergence warnings
    ))
])

# Training the model
# pipeline.fit(X_train, y_train)

# Checking the sparsity (How many features were set to exactly 0 by L1)
# model = pipeline.named_steps['clf']
# non_zero_features = sum(model.coef_[0] != 0)
# print(f"Features used out of 20: {non_zero_features}")

### Multi-Class Classification in SVM: The Battle of OVO vs OVR

#### 1. Basic Intuition (The Ground Reality)
Bhai, Support Vector Machine (SVM) jab banaya gaya tha, toh wo sirf **Binary Classifier** tha. Yani uska engine sirf do cheezon ko alag kar sakta hai (jaise "Cat" aur "Dog", ya 0 aur 1). 
Lekin real world me toh classification 3, 4 ya 100 categories ka hota hai (jaise Apple, Banana, Orange aur Mango). Ab ek akeli seedhi line (hyperplane) 4 cheezon ko ek sath kaise alag karegi?

Is problem ko solve karne ke liye, algorithm ke aage ek "Brain/Strategy" lagayi jati hai jo badi problem ko chhote-chhote Binary tasks me tod deti hai. Iske do sabse mashhoor desi tarike hain:
1. **OVO (One-vs-One) / "Dangal Mode":** Har class ka baki har class ke sath ek-ek match karao. (Apple vs Banana, Apple vs Orange, Banana vs Orange).
2. **OVR (One-vs-Rest) / "Me vs The World":** Ek class ko ek taraf rakho, aur baki sabko dushman maan lo. (Apple vs [Banana+Orange], Banana vs [Apple+Orange]).

---

#### 2. Core Concepts & Architecture (Step-by-Step Breakdown)

**A. SVC and NuSVC (`decision_function_shape`)**
SVC aur NuSVC under-the-hood `libsvm` engine par chalte hain. Aur `libsvm` multi-class ko handle karne ke liye historically **OVO (One-vs-One)** architecture ka use karta hai.

* **OVO ka architecture:** Agar aapke paas $K$ classes hain, toh model total $\frac{K(K-1)}{2}$ chote-chote binary SVM models banayega. (Example: 3 classes = 3 models, 10 classes = 45 models!).
* **`decision_function_shape` parameter ka asli sach (Crucial Engineer Fact):** Scikit-learn me default `decision_function_shape = 'ovr'` hota hai. Par iska matlab yeh nahi ki SVC OVR se train ho raha hai! SVC hamesha train OVO se hi hota hai, yeh parameter sirf output matrix (jo aapke code me print hota hai) ki "Shape" ko standard OVR format me convert karke dikhata hai taaki doosre sklearn algorithms ke sath compatibility bani rahe. Aap isko `'ovo'` set kar sakte hain agar aapko raw OVO voting results chahiye.

**B. LinearSVC (`multi_class`)**
LinearSVC backend me `liblinear` engine par chalta hai. Yeh highly scalable hai aur bade data ke liye bana hai.

* **`multi_class = 'ovr'` (Default):** Yeh directly **One-vs-Rest** architecture banata hai. Agar 10 classes hain, toh yeh strictly 10 SVM models banayega. Yeh OVO ke comparison me bohot fast aur memory-efficient hai.
* **`multi_class = 'crammer_singer'`:** Yeh OVO/OVR jaisa "hack" nahi hai. Crammer and Singer (two researchers) ne ek aisa advanced math function banaya jo ek hi single equation me sari classes ko ek sath optimize karta hai. **The Catch?** Yeh math me perfect hai, par computation me itna slow aur complex hai ki industry me practically isko kabhi use nahi kiya jata. 

---

#### 3. Advanced Mathematics (Behind the Scenes)

**How Prediction Works in OVO (Voting System):**
Jab aap ek naya data point model ko dete hain:
Har $\frac{K(K-1)}{2}$ classifier us point par predict karta hai aur apni winning class ko ek "Vote" deta hai. Jis class ko sabse zyada votes milte hain, wo final prediction hoti hai (Max Wins Rule).

**How Prediction Works in OVR (Confidence Score / Max Margin):**
Jab $K$ models banaye jate hain, toh har model ek equation deta hai: $f_i(x) = w_i \cdot x + b_i$.
Jab naya point aata hai, toh sabhi $K$ models apna score (boundary se distance) calculate karte hain. 
$$Final\_Class = \text{argmax}_{i \in \{1...K\}} (w_i \cdot x + b_i)$$
Yani jis model ne sabse confidence se kaha (boundary se sabse door point mila), wahi class jeetegi.

---

#### 4. Real-World Industry Use-Case
**E-commerce Support Ticket Routing (NLP):**
Maan lijiye Zomato ka customer support system hai. Har din 1 Lakh complaints aati hain, jinki 20 alag-alag categories hain ("Refund", "Missing Item", "Bad Quality", etc.).
Agar hum `SVC` (OVO) chalayenge, toh $20 \times 19 / 2 = 190$ models banenge. Yeh memory aur time dono crash kar dega.
Wahan hum text data (jisme waise bhi millions of features hote hain) ke sath `LinearSVC(multi_class='ovr')` use karte hain. Sirf 20 models banenge, fast matrix multiplication hoga, aur prediction 1 millisecond me routing server ko chali jayegi.

---

#### 5. Modern Implementation (Production Grade Code)
Production me jab multi-class problem hoti hai, toh OVO aur OVR ko carefully pipeline me implement kiya jata hai.


In [23]:

import numpy as np
from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Generating a dummy 3-class dataset
X, y = make_classification(n_samples=5000, n_features=20, n_informative=15, n_classes=3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# ---- Scenario 1: Using SVC (OVO under the hood, but returning OVO/OVR shape) ----
# Jab accuracy prime ho aur data medium size ka ho
svc_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(
        kernel='rbf',
        decision_function_shape='ovo', # Explicitly asking for One-vs-One raw output shape
        gamma='scale',
        C=1.0
    ))
])
# svc_pipeline.fit(X_train, y_train)


# ---- Scenario 2: Using LinearSVC (True OVR Architecture) ----
# Jab data massive ho (e.g., Millions of rows) aur speed chahiye
linear_svc_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LinearSVC(
        penalty='l2',
        loss='squared_hinge',
        multi_class='ovr',            # Explicitly setting One-vs-Rest (1 model per class)
        dual=False,                   # Mandatory kyuki samples > features
        max_iter=10000
    ))
])
# linear_svc_pipeline.fit(X_train, y_train)

### SVM in the Real World: The Ultimate Pros and Cons

#### 1. Basic Intuition (The Ground Reality)
Machine Learning me koi bhi ek algorithm perfect (God-algorithm) nahi hota. Har model ke apne superpowers aur apne weak points hote hain. SVM ek highly specialized "Sniper" ki tarah hai. Jab aapko ekdam precise, sharp, aur memory-efficient boundary chahiye, toh SVM se behtar kuch nahi. Lekin jab aapko speed me probabilities nikalni hon ya data me noise bohot zyada ho, toh yeh struggle karta hai.

Chaliye SVM ke in core behavior (fayde aur nuksan) ko ekdam engineering level par decode karte hain.

---

#### 2. The Advantages (Superpowers of SVM)

**A. King of High-Dimensional Spaces ($Dimensions > Samples$)**
* **Concept:** Aksar data me rows (samples) kam hote hain, par columns (features/dimensions) bohot zyada hote hain. Jaise Genetics me: Sirf 100 patients ka data (100 rows), par har patient ke 20,000 genes (20,000 columns). 
* **Why SVM shines:** Normal algorithms (jaise Decision Trees ya KNN) isme confuse ho jate hain ("Curse of Dimensionality"). Lekin SVM effectively kaam karta hai kyunki optimization ke waqt (Dual Problem me), yeh features ke size par depend nahi karta, balki data points ke aapas ke relations (dot products) par focus karta hai.

**B. Extreme Memory Efficiency (The Support Vector Magic)**
* **Concept:** Model train hone ke baad, SVM ko prediction karne ke liye pura data yaad rakhne ki zarurat nahi hoti. 
* **The Reality:** Agar aapne 10 Lakh (1 Million) rows par SVM train kiya hai, aur usme se sirf 5,000 points margin ke paas hain (Support Vectors). Toh training ke baad SVM baaki 9,95,000 data points ko **RAM se permanently delete** kar deta hai! Prediction sirf un 5,000 bache hue points ke base par hota hai. Isliye deployment me iska size bohot chhota (MBs me) hota hai.

**C. Unmatched Versatility (The Kernel Trick)**
* **Concept:** Linear, Polynomial, RBF, Sigmoid—SVM kisi bhi shape ke data me dhal sakta hai. Aap custom kernels bhi design kar sakte hain (jaise String kernels specifically DNA text ya English sentences ke liye).

---

#### 3. The Disadvantages (The Achilles' Heel of SVM)

**A. No Direct Probabilities (The Expensive 5-Fold Problem)**
* **Concept:** Logistic regression ya Random Forest aapko direct bata dete hain ki "80% chance hai ki yeh dog hai, 20% cat hai". SVM aisa **nahi** karta. SVM sirf yeh batata hai ki point boundary se kitni door hai (Distance like $2.5$ or $-1.3$).
* **The Hack (Platt Scaling):** Probabilities ($0$ to $1$) nikalne ke liye, Sklearn background me ek technique use karta hai jise **Platt Scaling** kehte hain. Isme model ko 5 alag-alag baar cross-validation me train karna padta hai, aur fir SVM ke distance output ke upar ek Logistic Regression lagaya jata hai.
* **Result:** Agar data thoda bhi bada hai, toh probabilities nikalne ke chakkar me SVM ki training speed **5-10 guna slow** ho jati hai.

**B. Overfitting Risk when Features $\gg$ Samples**
* **Concept:** Agar aapke paas 10,000 columns hain aur sirf 100 rows hain, toh data itna faila hua (sparse) hai ki wahan automatically ek seedhi line (Linear Kernel) aaram se fit ho sakti hai. 
* **The Mistake:** Aisi situation me agar aapne galti se complex kernel (jaise `'rbf'` ya `'poly'`) laga diya, toh model overfit ho jayega (har ek point ke charo taraf ek alag island bana lega). Aise case me hamesha regularized Linear kernel use karna chahiye.

---

#### 4. Advanced Mathematics (Platt Scaling)
SVM ke normal equation $f(x) = w \cdot x + b$ se probability nikalne ke liye, Platt equation yeh lagti hai:

$$P(y=1 | x) = \frac{1}{1 + \exp(A \cdot f(x) + B)}$$

Yahan $A$ aur $B$ naye parameters hain, jinhe dhoondhne ke liye algorithm ko ek extra internal model train karna padta hai (using Cross-Validation), jiske wajah se yeh computationally "expensive" ban jata hai.

---

#### 5. Modern Implementation (Production Grade Code)
Production me jab hum SVM likhte hain, toh in pros/cons ka dhyan rakhte hue kuch strict rules follow kiye jate hain. 



In [25]:

import time
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification

# Generating data where Features (2000) are greater than Samples (500)
X, y = make_classification(n_samples=500, n_features=2000, n_informative=100, random_state=42)

# ---- BEST PRACTICE 1: Handling D > N ----
# Kyunki columns rows se bohot zyada hain, RBF Kernel Overfit karega.
# Isliye hum strictly 'linear' kernel use karenge aur C ko chhota rakhenge (Strict Regularization).

# ---- BEST PRACTICE 2: Handling Probabilities ----
# probability=True karne se training bohot slow ho jayegi Platt Scaling ki wajah se.
# Production me agar probabilities zaroori nahi hain (sirf class chahiye), toh ise False rakhein.

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='linear',    # Mandatory linear for extremely high dimensions
        C=0.1,              # Strong regularization to avoid overfitting
        probability=True,   # Warning: This makes fit() run 5x slower!
        random_state=42
    ))
])

# Timing the execution to show the "Expensive" nature of probabilities
start_time = time.time()
pipeline.fit(X, y)
print(f"Training Time (with probability=True): {time.time() - start_time:.4f} seconds")

# Getting probability (not just distance)
# sample_prob = pipeline.predict_proba(X[:1])
# print(f"Probability Estimates: {sample_prob}")

Training Time (with probability=True): 0.5246 seconds
